In [1]:
# ==========================
# 1. Bibliotecas e estrutura
# ==========================

import csv
import platform
import re
from pathlib import Path

import duckdb
import pandas as pd

COLUNAS = [
    "ENTORNO", "PREFPE", "PRESPE", "PFEENT", "CODACT",
    "PCODCL", "PNOMCL", "PRUTA", "PCATCL", "CODTLI",
    "CODMOP", "PUNTOS", "PUNTOE", "PCAPIK", "CPOCLI",
    "DATM51", "JDEUNS", "JDECJS", "JDEPLS", "JDECPS",
    "JDERLS", "JDEKBS", "JDEKNS", "JDEUNE", "DATEXC",
    "UNIDAC",
]

COLUNA_DATA = "PFEENT"
COLUNA_ORIGEM = "ficheiro_origem"
COLUNAS_CHAVE = ("ENTORNO", "PREFPE", "PRESPE")
PREFIXO_TABELA = "ingresso_danone"
TAMANHO_BATCH = 10_000
REGEX_ANO = re.compile(r"^(\d{4})")


In [2]:
# ==========================
# 2. Caminhos
# ==========================

if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\Documents\python\00.DB\2026.duckdb"
    )
    PASTA_FICHEIROS = Path(
        r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27"
    )

elif platform.system() == "Darwin":
    BASE = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
    )
    DB_PATH = BASE / "00_DB" / "2026.duckdb"
    PASTA_FICHEIROS = BASE / "kilospo"

else:
    raise OSError(
        f"Sistema operativo não suportado: {platform.system()}"
    )

DB_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not PASTA_FICHEIROS.exists():
    raise FileNotFoundError(
        f"Pasta não encontrada: {PASTA_FICHEIROS}"
    )

with duckdb.connect(str(DB_PATH)) as con:
    con.execute("SELECT 1")

print(f"✓ BD: {DB_PATH}")
print(f"✓ Pasta: {PASTA_FICHEIROS}")


✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb
✓ Pasta: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/kilospo


In [3]:
# ==========================
# 3. Funções auxiliares
# ==========================

def obter_ano(valor):
    if valor is None:
        return None

    texto = str(valor).strip()
    correspondencia = REGEX_ANO.match(texto)

    if (
        not correspondencia
        or correspondencia.group(1) == "0000"
    ):
        return None

    return int(correspondencia.group(1))


def tabela_ano(ano):
    return f"{PREFIXO_TABELA}_{ano}"


def detectar_codificacao(caminho):
    for codificacao in (
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin-1",
    ):
        try:
            with caminho.open(
                "r",
                encoding=codificacao,
                newline="",
            ) as ficheiro:
                ficheiro.read(100_000)

            return codificacao

        except UnicodeDecodeError:
            continue

    raise UnicodeError(
        f"Não foi possível identificar a codificação: {caminho.name}"
    )


def normalizar_cabecalho(nome):
    return str(nome).strip()


def normalizar_valor(valor):
    if valor is None:
        return None

    texto = str(valor).strip()

    return texto if texto else None


In [4]:
# ==========================
# 4. Estrutura DuckDB
# ==========================

def criar_tabela(con, ano):
    tabela = tabela_ano(ano)

    colunas_sql = ", ".join(
        f'"{coluna}" VARCHAR'
        for coluna in COLUNAS
    )

    con.execute(f"""
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id BIGINT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" VARCHAR NOT NULL
        )
    """)

    con.execute(
        f'CREATE UNIQUE INDEX IF NOT EXISTS '
        f'"idx_{tabela}_chave" '
        f'ON "{tabela}" '
        f'("ENTORNO", "PREFPE", "PRESPE")'
    )

    con.execute(
        f'CREATE INDEX IF NOT EXISTS '
        f'"idx_{tabela}_pfeent" '
        f'ON "{tabela}" ("PFEENT")'
    )

    con.execute(
        f'CREATE INDEX IF NOT EXISTS '
        f'"idx_{tabela}_pcodcl" '
        f'ON "{tabela}" ("PCODCL")'
    )

    return tabela


def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(
        con,
        ano,
    )

    antes = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    max_id = con.execute(f"""
        SELECT COALESCE(
            MAX(TRY_CAST(id AS BIGINT)),
            0
        )
        FROM "{tabela}"
    """).fetchone()[0]

    nomes_colunas = [
        *COLUNAS,
        COLUNA_ORIGEM,
    ]

    df_batch = pd.DataFrame(
        linhas,
        columns=nomes_colunas,
        dtype=object,
    )

    df_batch.insert(
        0,
        "id",
        range(
            int(max_id) + 1,
            int(max_id) + 1 + len(df_batch),
        ),
    )

    con.register(
        "batch_kilospo",
        df_batch,
    )

    try:
        con.execute(f"""
            INSERT OR IGNORE INTO "{tabela}"
            SELECT *
            FROM batch_kilospo
        """)

    finally:
        con.unregister(
            "batch_kilospo"
        )

    depois = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    inseridos = depois - antes
    duplicados = len(linhas) - inseridos

    return inseridos, duplicados


In [5]:
# ==========================
# 5. Importação dos CSV
# ==========================

ficheiros = sorted(
    caminho
    for caminho in PASTA_FICHEIROS.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower() == ".csv"
    )
)

if not ficheiros:
    raise FileNotFoundError(
        f"Nenhum ficheiro CSV em: {PASTA_FICHEIROS}"
    )

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "sem_chave": 0,
    "erros": 0,
}

with duckdb.connect(str(DB_PATH)) as con:

    for numero, caminho in enumerate(
        ficheiros,
        start=1,
    ):
        transacao_aberta = False

        try:
            codificacao = detectar_codificacao(
                caminho
            )

            estatisticas = {
                "linhas": 0,
                "novos": 0,
                "duplicados": 0,
                "sem_data": 0,
                "sem_chave": 0,
            }

            batches = {}

            con.begin()
            transacao_aberta = True

            with caminho.open(
                "r",
                encoding=codificacao,
                newline="",
            ) as ficheiro:

                reader = csv.DictReader(
                    ficheiro,
                    delimiter=";",
                )

                if reader.fieldnames is None:
                    raise ValueError(
                        "CSV sem cabeçalho."
                    )

                reader.fieldnames = [
                    normalizar_cabecalho(coluna)
                    for coluna in reader.fieldnames
                ]

                colunas_em_falta = [
                    coluna
                    for coluna in (
                        *COLUNAS_CHAVE,
                        COLUNA_DATA,
                    )
                    if coluna not in reader.fieldnames
                ]

                if colunas_em_falta:
                    raise ValueError(
                        "Colunas obrigatórias em falta: "
                        f"{colunas_em_falta}"
                    )

                for registo in reader:

                    estatisticas["linhas"] += 1

                    registo = {
                        normalizar_cabecalho(coluna):
                            normalizar_valor(valor)
                        for coluna, valor in registo.items()
                        if coluna is not None
                    }

                    ano = obter_ano(
                        registo.get(COLUNA_DATA)
                    )

                    if ano is None:
                        estatisticas["sem_data"] += 1
                        continue

                    if any(
                        not registo.get(coluna)
                        for coluna in COLUNAS_CHAVE
                    ):
                        estatisticas["sem_chave"] += 1
                        continue

                    linha = [
                        registo.get(coluna)
                        for coluna in COLUNAS
                    ]

                    linha.append(
                        caminho.name
                    )

                    batches.setdefault(
                        ano,
                        [],
                    ).append(linha)

                    if (
                        len(batches[ano])
                        >= TAMANHO_BATCH
                    ):
                        inseridos, duplicados = inserir_linhas(
                            con,
                            ano,
                            batches[ano],
                        )

                        estatisticas["novos"] += inseridos
                        estatisticas["duplicados"] += duplicados

                        totais["inseridos"][ano] = (
                            totais["inseridos"].get(
                                ano,
                                0,
                            )
                            + inseridos
                        )

                        batches[ano].clear()

            for ano, batch in sorted(
                batches.items()
            ):
                inseridos, duplicados = inserir_linhas(
                    con,
                    ano,
                    batch,
                )

                estatisticas["novos"] += inseridos
                estatisticas["duplicados"] += duplicados

                totais["inseridos"][ano] = (
                    totais["inseridos"].get(
                        ano,
                        0,
                    )
                    + inseridos
                )

            con.commit()
            transacao_aberta = False

            totais["ficheiros"] += 1
            totais["linhas"] += estatisticas["linhas"]
            totais["duplicados"] += estatisticas["duplicados"]
            totais["sem_data"] += estatisticas["sem_data"]
            totais["sem_chave"] += estatisticas["sem_chave"]

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} | "
                f"+{estatisticas['novos']:,} novos | "
                f"{estatisticas['duplicados']:,} duplicados | "
                f"{estatisticas['sem_data']:,} sem data | "
                f"{estatisticas['sem_chave']:,} sem chave"
            )

        except Exception as erro:

            if transacao_aberta:
                con.rollback()

            totais["erros"] += 1

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{caminho.name} — ERRO: {erro}"
            )

    con.checkpoint()


# ==========================
# 6. Resumo
# ==========================

print("\n--- RESUMO ---")

print(
    f"Ficheiros: {totais['ficheiros']} | "
    f"Linhas lidas: {totais['linhas']:,} | "
    f"Duplicados: {totais['duplicados']:,} | "
    f"Sem data: {totais['sem_data']:,} | "
    f"Sem chave: {totais['sem_chave']:,} | "
    f"Erros: {totais['erros']}"
)

for ano, quantidade in sorted(
    totais["inseridos"].items()
):
    print(
        f"Linhas novas em {tabela_ano(ano)}: "
        f"{quantidade:,}"
    )


[1/2] INFKILP (1).CSV | +1,447 novos | 3,712 duplicados | 1 sem data | 0 sem chave
[2/2] INFKILP.CSV | +471 novos | 1,297 duplicados | 1 sem data | 0 sem chave

--- RESUMO ---
Ficheiros: 2 | Linhas lidas: 6,929 | Duplicados: 5,009 | Sem data: 2 | Sem chave: 0 | Erros: 0
Linhas novas em ingresso_danone_2026: 1,918


In [6]:
# ==========================
# 7. Validação final
# ==========================

with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    tabelas = con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name LIKE 'ingresso_danone_%'
        ORDER BY table_name
    """).fetchall()

    validacao = []
    duplicados = []

    for (tabela,) in tabelas:

        total, ids_unicos = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT id)
            FROM "{tabela}"
        """).fetchone()

        grupos_duplicados, linhas_duplicadas = con.execute(f"""
            SELECT
                COUNT(*),
                COALESCE(SUM(n), 0)
            FROM (
                SELECT
                    "ENTORNO",
                    "PREFPE",
                    "PRESPE",
                    COUNT(*) AS n
                FROM "{tabela}"
                GROUP BY
                    "ENTORNO",
                    "PREFPE",
                    "PRESPE"
                HAVING COUNT(*) > 1
            )
        """).fetchone()

        validacao.append({
            "Tabela": tabela,
            "Linhas": total,
            "IDs_Unicos": ids_unicos,
            "Grupos_Duplicados": grupos_duplicados,
            "Linhas_Duplicadas": linhas_duplicadas,
            "OK": (
                total == ids_unicos
                and grupos_duplicados == 0
            ),
        })

        if grupos_duplicados > 0:

            df_dup = con.execute(f"""
                WITH chaves_duplicadas AS (
                    SELECT
                        "ENTORNO",
                        "PREFPE",
                        "PRESPE",
                        COUNT(*) AS qtd_duplicados
                    FROM "{tabela}"
                    GROUP BY
                        "ENTORNO",
                        "PREFPE",
                        "PRESPE"
                    HAVING COUNT(*) > 1
                )

                SELECT
                    d.qtd_duplicados,
                    t.*
                FROM "{tabela}" t

                INNER JOIN chaves_duplicadas d
                    ON t."ENTORNO"
                        IS NOT DISTINCT FROM d."ENTORNO"
                   AND t."PREFPE"
                        IS NOT DISTINCT FROM d."PREFPE"
                   AND t."PRESPE"
                        IS NOT DISTINCT FROM d."PRESPE"

                ORDER BY
                    t."ENTORNO",
                    t."PREFPE",
                    t."PRESPE",
                    TRY_CAST(t.id AS BIGINT)
            """).df()

            df_dup.insert(
                0,
                "Tabela",
                tabela,
            )

            duplicados.append(
                df_dup
            )

df_validacao = pd.DataFrame(
    validacao
)

df_duplicados = (
    pd.concat(
        duplicados,
        ignore_index=True,
    )
    if duplicados
    else pd.DataFrame()
)

df_validacao


,Tabela,Linhas,IDs_Unicos,Grupos_Duplicados,Linhas_Duplicadas,OK
0,ingresso_danone_2026,57990,57990,0,0,True


In [7]:
# ==========================
# 8. Ligação para análises
# ==========================

con = duckdb.connect(
    str(DB_PATH),
    read_only=True,
)

print(f"✓ BD ligada: {DB_PATH}")


✓ BD ligada: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb


In [8]:
# ==========================
# 9. Peso preparado
# ==========================

query = """
SELECT
    SUBSTR(DATEXC, 1, 6) AS Mes,
    SUM(
        TRY_CAST(
            REPLACE(JDEKNS, ',', '.')
            AS DOUBLE
        )
    ) AS Total_JDEKNS
FROM ingresso_danone_2026
GROUP BY SUBSTR(DATEXC, 1, 6)
ORDER BY Mes
"""

df_jdekns = con.execute(
    query
).df()

df_jdekns["Total_JDEKNS"] = (
    df_jdekns["Total_JDEKNS"]
    .round(0)
    .astype("Int64")
)

df_jdekns


,Mes,Total_JDEKNS
0,0,33723
1,202512,293293
2,202601,4921542
3,202602,4343495
4,202603,5050950
5,202604,5202758
6,202605,4921021
7,202606,5409947
8,202607,6230968
9,202608,5168994


In [9]:
# ==========================
# 10. Paletes
# ==========================

query = """
SELECT
    SUBSTR(FENTREGA, 1, 6) AS Mes,
    SUM(
        TRY_CAST(
            REPLACE(PALETS, ',', '.')
            AS DOUBLE
        )
    ) AS Total_Paletes
FROM inform_27_2026
WHERE CODACT = '011'
GROUP BY SUBSTR(FENTREGA, 1, 6)
ORDER BY Mes
"""

df_paletes = con.execute(
    query
).df()

df_paletes["Total_Paletes"] = (
    df_paletes["Total_Paletes"]
    .round(0)
    .astype("Int64")
)

df_paletes


,Mes,Total_Paletes
0,202601,14957
1,202602,13015
2,202603,14462
3,202604,15011
4,202605,16658
5,202606,16042
6,202607,17007
7,202608,18342
8,202609,3509


In [10]:
# ==========================
# 11. Comparação
# ==========================

df_combined = (
    df_jdekns
    .merge(
        df_paletes,
        on="Mes",
        how="outer",
    )
    .sort_values("Mes")
    .rename(
        columns={
            "Mes": "Ano_Mes",
            "Total_JDEKNS": "Peso_Preparado",
            "Total_Paletes": "Numero_Paletes",
        }
    )
    [
        [
            "Ano_Mes",
            "Numero_Paletes",
            "Peso_Preparado",
        ]
    ]
)

df_combined


,Ano_Mes,Numero_Paletes,Peso_Preparado
0,0,<NA>,33723
1,202512,<NA>,293293
2,202601,14957,4921542
3,202602,13015,4343495
4,202603,14462,5050950
5,202604,15011,5202758
6,202605,16658,4921021
7,202606,16042,5409947
8,202607,17007,6230968
9,202608,18342,5168994
